# Notebook 03 — Customer Feature Engineering

**Goal:** Convert the customer-level profile from Notebook 02 into a clean, behaviour-focused feature matrix for clustering.

We will:
- Load `processed/customer_profile.csv` from Google Drive.
- Create interpretable behavioural features (engagement, activity, transaction behaviour, recency and campaign response).
- Keep billing/overdue information separate from behavioural clustering.
- Remove identifiers and leakage/operational fields.
- Handle missing values and skewed numeric distributions.
- Export the feature matrix for Notebook 04.

**Important:** We are not assigning Loyal/Dormant/Overdue labels here. Clustering will discover behavioural groups. Billing status remains a separate operational attribute.

## 1. Mount Google Drive and locate the profile

The code checks the expected folder first. If the profile is missing, it looks for the ZIP in `MyDrive`, extracts it, and then checks again.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile

MYDRIVE = Path('/content/drive/MyDrive')
DATA_FOLDER_NAME = 'customer_loyalty_synthetic_20k'
DATA_DIR = MYDRIVE / DATA_FOLDER_NAME
ZIP_PATH = MYDRIVE / f'{DATA_FOLDER_NAME}.zip'

if not (DATA_DIR / 'processed' / 'customer_profile.csv').exists():
    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            f'Could not find {DATA_DIR / "processed" / "customer_profile.csv"} '
            f'or {ZIP_PATH}. Run Notebook 02 first and confirm the ZIP is in MyDrive.'
        )
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)

if not (DATA_DIR / 'processed' / 'customer_profile.csv').exists():
    matches = list(DATA_DIR.rglob('customer_profile.csv'))
    if not matches:
        raise FileNotFoundError(
            'customer_profile.csv was not found. Run Notebook 02 and check its output path.'
        )
    PROFILE_PATH = matches[0]
    DATA_DIR = PROFILE_PATH.parents[1]
else:
    PROFILE_PATH = DATA_DIR / 'processed' / 'customer_profile.csv'

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Profile file:', PROFILE_PATH)

Mounted at /content/drive
Profile file: /content/drive/MyDrive/customer_loyalty_synthetic_20k/customer_loyalty_synthetic_20k/processed/customer_profile.csv


## 2. Load and inspect the customer profile

In [2]:
import pandas as pd
import numpy as np

profile = pd.read_csv(PROFILE_PATH)
print('Shape:', profile.shape)
print('Unique customers:', profile['customer_id'].nunique())
display(profile.head())
print('\nColumns:')
print(profile.columns.tolist())

Shape: (20000, 38)
Unique customers: 20000


,customer_id,join_date,tenure_months,primary_product,customer_status,preferred_channel,marketing_consent,monthly_contact_cap,statement_cycle,has_loan,...,campaigns_received,campaigns_accepted,campaign_acceptance_rate,communication_count,delivered_count,opened_count,clicked_count,communication_delivery_rate,communication_engagement_rate,transaction_recency_days
0,CUST000001,2026-07-04,4,Premium,New,App,True,4,Monthly,False,...,4.0,0.0,0.0,3.0,2.0,1.0,0.0,0.666667,0.333333,54.0
1,CUST000002,2023-09-18,38,Current,Active,Email,True,4,Monthly,False,...,5.0,2.0,0.4,2.0,2.0,1.0,0.0,1.000000,0.500000,37.0
2,CUST000003,2026-05-05,6,Salary,Active,App,True,5,Monthly,False,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.000000,0.000000,32.0
3,CUST000004,2026-07-04,4,Savings,New,Email,True,2,Monthly,False,...,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.000000,0.000000,62.0
4,CUST000005,2024-08-13,27,Salary,Active,WhatsApp,True,4,Monthly,False,...,4.0,2.0,0.5,3.0,3.0,2.0,2.0,1.000000,0.666667,16.0



Columns:
['customer_id', 'join_date', 'tenure_months', 'primary_product', 'customer_status', 'preferred_channel', 'marketing_consent', 'monthly_contact_cap', 'statement_cycle', 'has_loan', 'observed_months', 'avg_active_days', 'avg_login_count', 'avg_app_sessions', 'avg_service_usage', 'avg_engagement_score', 'latest_engagement_score', 'successful_transaction_count', 'total_transaction_value_inr', 'avg_transaction_value_inr', 'median_transaction_value_inr', 'last_transaction_date', 'bill_count', 'total_amount_due_inr', 'total_amount_paid_inr', 'overdue_bill_count', 'avg_days_overdue', 'max_days_overdue', 'campaigns_received', 'campaigns_accepted', 'campaign_acceptance_rate', 'communication_count', 'delivered_count', 'opened_count', 'clicked_count', 'communication_delivery_rate', 'communication_engagement_rate', 'transaction_recency_days']


## 3. Select behaviour-related inputs

We use features that describe *how customers engage and transact*:
- Activity: active days, logins, app sessions, service usage, engagement.
- Transactions: frequency, monetary value and recency.
- Campaigns: acceptance rate.
- Tenure: customer relationship duration.

We deliberately exclude:
- `customer_id` and other identifiers.
- Billing/overdue fields. These are retained separately for later business rules, not clustering.
- Consent, contact caps and communication controls.
- Synthetic validation labels.

In [3]:
candidate_features = [
    'tenure_months',
    'observed_months',
    'avg_active_days',
    'avg_login_count',
    'avg_app_sessions',
    'avg_service_usage',
    'avg_engagement_score',
    'latest_engagement_score',
    'successful_transaction_count',
    'total_transaction_value_inr',
    'avg_transaction_value_inr',
    'median_transaction_value_inr',
    'transaction_recency_days',
    'campaigns_received',
    'campaign_acceptance_rate',
]

available_features = [c for c in candidate_features if c in profile.columns]
missing_candidates = [c for c in candidate_features if c not in profile.columns]

print('Features available:', available_features)
print('Features not present in profile:', missing_candidates)

if len(available_features) < 5:
    raise ValueError('Too few expected behavioural features were found. Check Notebook 02 output.')

Features available: ['tenure_months', 'observed_months', 'avg_active_days', 'avg_login_count', 'avg_app_sessions', 'avg_service_usage', 'avg_engagement_score', 'latest_engagement_score', 'successful_transaction_count', 'total_transaction_value_inr', 'avg_transaction_value_inr', 'median_transaction_value_inr', 'transaction_recency_days', 'campaigns_received', 'campaign_acceptance_rate']
Features not present in profile: []


## 4. Create additional interpretable behavioural features

In [4]:
features = profile[['customer_id'] + available_features].copy()

# Ratios and normalised measures make behaviour easier to compare across customers.
if {'avg_active_days', 'observed_months'}.issubset(features.columns):
    # Approximate active-day rate: average active days divided by 30 days.
    features['active_day_rate'] = (features['avg_active_days'] / 30).clip(0, 1)

if {'avg_login_count', 'avg_app_sessions'}.issubset(features.columns):
    features['sessions_per_login'] = (
        features['avg_app_sessions'] / features['avg_login_count'].replace(0, np.nan)
    )

if {'campaigns_accepted', 'campaigns_received'}.issubset(profile.columns):
    features['campaign_acceptance_rate'] = (
        profile['campaigns_accepted'] /
        profile['campaigns_received'].replace(0, np.nan)
    )

if {'successful_transaction_count', 'observed_months'}.issubset(features.columns):
    features['transactions_per_observed_month'] = (
        features['successful_transaction_count'] /
        features['observed_months'].replace(0, np.nan)
    )

if {'total_transaction_value_inr', 'successful_transaction_count'}.issubset(features.columns):
    features['spend_per_transaction_inr'] = (
        features['total_transaction_value_inr'] /
        features['successful_transaction_count'].replace(0, np.nan)
    )

display(features.head())

,customer_id,tenure_months,observed_months,avg_active_days,avg_login_count,avg_app_sessions,avg_service_usage,avg_engagement_score,latest_engagement_score,successful_transaction_count,total_transaction_value_inr,avg_transaction_value_inr,median_transaction_value_inr,transaction_recency_days,campaigns_received,campaign_acceptance_rate,active_day_rate,sessions_per_login,transactions_per_observed_month,spend_per_transaction_inr
0,CUST000001,4,12,2.250000,4.250000,2.416667,1.333333,43.491667,45.8,9.0,2367.32,263.035556,180.500,54.0,4.0,0.0,0.075000,0.568627,0.750000,263.035556
1,CUST000002,38,12,11.166667,14.666667,7.333333,6.583333,48.483333,48.8,10.0,9320.91,932.091000,795.700,37.0,5.0,0.4,0.372222,0.500000,0.833333,932.091000
2,CUST000003,6,12,15.750000,20.750000,14.166667,12.750000,40.666667,47.1,10.0,3283.54,328.354000,230.045,32.0,1.0,0.0,0.525000,0.682731,0.833333,328.354000
3,CUST000004,4,12,8.916667,13.416667,8.833333,6.666667,41.800000,49.2,9.0,1753.22,194.802222,162.970,62.0,1.0,0.0,0.297222,0.658385,0.750000,194.802222
4,CUST000005,27,12,27.750000,41.916667,26.583333,21.750000,80.083333,72.1,13.0,1931.74,148.595385,122.570,16.0,4.0,0.5,0.925000,0.634195,1.083333,148.595385


## 5. Check missing values and handle them

A missing transaction or campaign aggregate often means the customer had no recorded event in that table. For count/value features, we use zero where that interpretation is valid. For rates and ratios, missing values can mean 'not applicable' (for example, no campaign received); we fill those with zero for clustering and document that choice.

In [5]:
feature_cols = [c for c in features.columns if c != 'customer_id']

missing_report = (
    features[feature_cols].isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename('missing_pct')
    .reset_index()
)
missing_report.columns = ['feature', 'missing_pct']
display(missing_report.round(2))

# All selected fields are numeric measures. Convert safely.
for col in feature_cols:
    features[col] = pd.to_numeric(features[col], errors='coerce')

# Replace infinities from divisions by zero, then fill undefined/no-event values with 0.
features[feature_cols] = features[feature_cols].replace([np.inf, -np.inf], np.nan)
features[feature_cols] = features[feature_cols].fillna(0)

print('Remaining missing values:', int(features[feature_cols].isna().sum().sum()))

,feature,missing_pct
0,campaign_acceptance_rate,4.15
1,total_transaction_value_inr,0.80
2,median_transaction_value_inr,0.80
3,spend_per_transaction_inr,0.80
4,transaction_recency_days,0.80
5,avg_transaction_value_inr,0.80
6,sessions_per_login,0.04
7,avg_active_days,0.00
8,observed_months,0.00
9,tenure_months,0.00


Remaining missing values: 0


## 6. Inspect skew and apply log transformation to highly skewed non-negative features

In [6]:
from IPython.display import display

skewness = features[feature_cols].skew(numeric_only=True).sort_values(ascending=False)
display(skewness.rename('skewness').to_frame().round(2))

# Log1p reduces the influence of very large values while preserving zero values.
log_candidates = [
    'total_transaction_value_inr',
    'avg_transaction_value_inr',
    'median_transaction_value_inr',
    'successful_transaction_count',
    'campaigns_received',
    'avg_login_count',
    'avg_app_sessions',
    'avg_service_usage',
]
log_applied = []
for col in log_candidates:
    if col in features.columns and (features[col] >= 0).all():
        features[col] = np.log1p(features[col])
        log_applied.append(col)

print('Applied log1p to:', log_applied)

,skewness
median_transaction_value_inr,9.47
avg_transaction_value_inr,7.30
spend_per_transaction_inr,7.30
sessions_per_login,6.63
transaction_recency_days,2.51
total_transaction_value_inr,2.22
campaign_acceptance_rate,0.83
avg_service_usage,0.55
avg_app_sessions,0.55
campaigns_received,0.54


Applied log1p to: ['total_transaction_value_inr', 'avg_transaction_value_inr', 'median_transaction_value_inr', 'successful_transaction_count', 'campaigns_received', 'avg_login_count', 'avg_app_sessions', 'avg_service_usage']


## 7. Remove constant features and validate the matrix

Constant features cannot help distinguish customers, so we remove them. We retain the customer ID in a separate mapping file and do not pass it to the clustering algorithm.

In [7]:
feature_cols = [c for c in features.columns if c != 'customer_id']

constant_cols = [c for c in feature_cols if features[c].nunique(dropna=False) <= 1]
print('Constant columns removed:', constant_cols)

features = features.drop(columns=constant_cols)
feature_cols = [c for c in features.columns if c != 'customer_id']

X = features[feature_cols].copy()

assert X.select_dtypes(exclude=[np.number]).empty, 'Non-numeric features remain.'
assert not X.isna().any().any(), 'Missing values remain.'
assert np.isfinite(X.to_numpy()).all(), 'Infinite values remain.'

print('Final feature matrix shape:', X.shape)
print('Number of customers:', features['customer_id'].nunique())
display(X.describe().T.round(2))

Constant columns removed: ['observed_months']
Final feature matrix shape: (20000, 18)
Number of customers: 20000


,count,mean,std,min,25%,50%,75%,max
tenure_months,20000.0,51.11,35.86,1.0,18.00,49.00,80.00,120.00
avg_active_days,20000.0,12.67,8.41,0.0,6.17,11.08,19.75,30.00
avg_login_count,20000.0,2.66,0.87,0.0,2.28,2.80,3.35,4.12
avg_app_sessions,20000.0,2.27,0.81,0.0,1.88,2.40,2.93,3.66
avg_service_usage,20000.0,2.12,0.78,0.0,1.73,2.23,2.76,3.53
avg_engagement_score,20000.0,55.04,27.59,0.0,37.98,54.61,81.47,99.82
latest_engagement_score,20000.0,54.37,29.13,0.0,34.80,54.80,80.10,100.00
successful_transaction_count,20000.0,2.30,0.64,0.0,1.95,2.40,2.77,3.58
total_transaction_value_inr,20000.0,7.79,1.15,0.0,7.34,7.98,8.48,10.72
avg_transaction_value_inr,20000.0,5.63,0.74,0.0,5.29,5.60,6.00,9.32


## 8. Keep billing status separate for business interpretation

Billing information is useful for the later decision layer, but it should not define behavioural clusters. We save a separate customer-to-billing reference table for use after clustering.

In [8]:
billing_cols = [
    c for c in [
        'customer_id', 'bill_count', 'total_amount_due_inr',
        'total_amount_paid_inr', 'overdue_bill_count',
        'avg_days_overdue', 'max_days_overdue'
    ] if c in profile.columns
]
billing_reference = profile[billing_cols].copy() if billing_cols else pd.DataFrame()

if not billing_reference.empty:
    billing_reference.to_csv(PROCESSED_DIR / 'customer_billing_reference.csv', index=False)
    print('Saved billing reference:', PROCESSED_DIR / 'customer_billing_reference.csv')
else:
    print('No billing columns found in profile; billing reference not created.')

Saved billing reference: /content/drive/MyDrive/customer_loyalty_synthetic_20k/customer_loyalty_synthetic_20k/processed/customer_billing_reference.csv


## 9. Export outputs for Notebook 04

In [9]:
# Keep the ID mapping separate from the numerical matrix.
customer_ids = features[['customer_id']].copy()
customer_ids.to_csv(PROCESSED_DIR / 'customer_id_mapping.csv', index=False)

X.to_csv(PROCESSED_DIR / 'customer_features_unscaled.csv', index=False)

feature_dictionary = pd.DataFrame({
    'feature': feature_cols,
    'description': [
        'Behavioural/customer measure derived from the customer profile; inspect source column and feature engineering cells.'
        for _ in feature_cols
    ]
})
feature_dictionary.to_csv(PROCESSED_DIR / 'customer_feature_dictionary.csv', index=False)

print('Saved:')
print('-', PROCESSED_DIR / 'customer_features_unscaled.csv')
print('-', PROCESSED_DIR / 'customer_id_mapping.csv')
print('-', PROCESSED_DIR / 'customer_feature_dictionary.csv')
print('Feature count:', len(feature_cols))

Saved:
- /content/drive/MyDrive/customer_loyalty_synthetic_20k/customer_loyalty_synthetic_20k/processed/customer_features_unscaled.csv
- /content/drive/MyDrive/customer_loyalty_synthetic_20k/customer_loyalty_synthetic_20k/processed/customer_id_mapping.csv
- /content/drive/MyDrive/customer_loyalty_synthetic_20k/customer_loyalty_synthetic_20k/processed/customer_feature_dictionary.csv
Feature count: 18


## 10. Handoff to Notebook 04 — Customer Segmentation

Notebook 04 will:
1. Load `customer_features_unscaled.csv`.
2. Standardise the features using `StandardScaler`.
3. Fit clustering models (starting with K-Means).
4. Select a candidate number of clusters using elbow/inertia and silhouette score.
5. Save cluster assignments separately and profile the resulting clusters.

**Do not interpret cluster numbers as labels yet.** Cluster `0`, `1`, etc. are arbitrary IDs. We will interpret them using feature profiles in the next notebook.